# Template Tuning with DimRed API

This notebook demonstrates how to run prompt tuning with template variables using the DimRed API.

## Overview

The workflow:
1. Load template dataset from `template_optimization_example.json`
2. Create a project and dataset
3. Add datapoints with template_var payloads
4. Create a prompt with template variable placeholders (e.g., `{{customer_name}}`)
5. Create a metric
6. Run tuning
7. Poll for results

## Setup

In [ ]:
import json
import logging
import sys
import os
from pathlib import Path

# Add parent directory to path to import client
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from client import DimRedAPIClient

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    stream=sys.stdout,
    force=True
)
logger = logging.getLogger(__name__)

## Configuration

In [ ]:
# API Configuration
API_KEY = os.environ.get("DIMRED_API_KEY")
BASE_URL = "https://api.dimred.com"

# Path to template dataset
TEMPLATE_DATASET_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'template_optimization_example.json'))

# Initialize client
client = DimRedAPIClient(API_KEY, BASE_URL)
print(f"✓ Initialized DimRed API client")
print(f"✓ Dataset path: {TEMPLATE_DATASET_PATH}")

## Step 1: Load Template Dataset

The dataset contains datapoints with template variable payloads. Each datapoint has:
- `input`: The article snippet to analyze
- `expected`: The expected response (is_perpetrator and reasoning)
- `payloads`: Array of template_var payloads with key/value pairs for substitution

In [ ]:
# Load the template dataset
with open(TEMPLATE_DATASET_PATH, 'r') as f:
    template_dataset = json.load(f)

print(f"Loaded {len(template_dataset)} datapoints from {TEMPLATE_DATASET_PATH}")
print(f"\nFirst datapoint structure:")
print(f"  - input keys: {list(template_dataset[0]['input'].keys())}")
print(f"  - expected keys: {list(template_dataset[0]['expected'].keys())}")
print(f"  - Number of payloads: {len(template_dataset[0]['payloads'])}")
print(f"  - Payload type: {template_dataset[0]['payloads'][0]['payload_type']}")
print(f"  - Template variable: {template_dataset[0]['payloads'][0]['payload']['key']} = {template_dataset[0]['payloads'][0]['payload']['value']}")

## Step 2: Create Project

In [ ]:
project_id = client.create_project(
    project_name="Perpetrator Classification Tuning",
    project_description="Testing template-based prompt tuning for financial crime perpetrator identification"
)

print(f"✓ Created project: {project_id}")

## Step 3: Create Dataset

In [ ]:
dataset_id = client.create_dataset(
    project_id=project_id,
    dataset_name="Perpetrator Classification Dataset"
)

print(f"✓ Created dataset: {dataset_id}")

## Step 4: Add Datapoints with Template Variable Payloads

When adding datapoints with template variables, the API expects:
- `input_data`: JSON string of the input
- `expected_output`: JSON string of the expected output
- `payloads`: Array of payload objects with structure:
  ```json
  {
    "payload_type": "template_var",
    "payload": {
      "key": "customer_name",
      "value": "John Doe"
    }
  }
  ```

The system will replace `{{customer_name}}` in the prompt with the provided value.

In [ ]:
# Convert dataset to API format
# Rename 'input' -> 'input_data' and 'expected' -> 'expected_output'
datapoints = []
for item in template_dataset:
    datapoint = {
        "input_data": json.dumps(item["input"]),
        "expected_output": json.dumps(item["expected"]),
        "payloads": item["payloads"]  # Pass payloads as-is
    }
    datapoints.append(datapoint)

# Add to dataset
count = client.add_datapoints(dataset_id, datapoints)
print(f"✓ Added {count} datapoints with template_var payloads")

# Display an example datapoint structure
print("\n=== Example Datapoint Structure ===")
example = datapoints[0]
print(f"\ninput_data: {example['input_data']}")
print(f"\nexpected_output: {example['expected_output']}")
print(f"\npayloads: {len(example['payloads'])} payload(s)")
print(f"\nPayload details:")
for i, payload in enumerate(example['payloads'], 1):
    print(f"  Payload {i}:")
    print(f"    - Type: {payload['payload_type']}")
    print(f"    - Template var: {payload['payload']['key']} = {payload['payload']['value']}")

## Step 5: Create Prompt

Create a prompt for perpetrator classification with template variable placeholders.
The `{{customer_name}}` placeholder will be replaced with the actual value from the payload.

In [ ]:
messages = [
    {
        "prompt_text": (
            "You are an expert analyst evaluating whether individuals are perpetrators in financial crime cases. "
            "Your task is to analyze an article snippet and determine if the person named '{{customer_name}}' "
            "is a perpetrator based on the evidence provided.\n\n"
            "Guidelines:\n"
            "- Consider formal charges, arrests, and documented evidence\n"
            "- Distinguish between perpetrators and cooperating witnesses\n"
            "- Account for presumption of innocence when investigations are ongoing\n"
            "- Look for concrete evidence vs. circumstantial patterns\n\n"
            "Respond with JSON containing:\n"
            "- is_perpetrator: true or false\n"
            "- reasoning: detailed explanation of your decision based on the evidence"
        ),
        "prompt_message_type": "system"
    }
]

# Output schema for structured response
output_schema = {
    "type": "object",
    "properties": {
        "is_perpetrator": {
            "type": "boolean",
            "description": "Whether the specified customer_name is determined to be a perpetrator"
        },
        "reasoning": {
            "type": "string",
            "description": "Detailed explanation of the decision based on available evidence"
        }
    },
    "required": ["is_perpetrator", "reasoning"],
    "additionalProperties": False
}

prompt_id = client.create_prompt(
    project_id=project_id,
    messages=messages,
    name="Perpetrator Classification with Template Variables",
    output_schema=output_schema
)

print(f"✓ Created prompt: {prompt_id}")
print(f"\nPrompt uses template variable: {{{{customer_name}}}}")

## Step 6: Create Metric

In [ ]:
metric_code = '''
import json

def metric_func(output, expected):
    """
    Check if the model correctly identified whether the person is a perpetrator.
    Returns 1.0 for correct classification, 0.0 for incorrect.
    """
    # Parse output and expected if they're strings
    if isinstance(output, str):
        try:
            output = json.loads(output)
        except json.JSONDecodeError:
            return 0.0

    if isinstance(expected, str):
        try:
            expected = json.loads(expected)
        except json.JSONDecodeError:
            return 0.0

    # Extract is_perpetrator field
    output_value = output.get("is_perpetrator")
    expected_value = expected.get("is_perpetrator")

    # Both must be present and match
    if output_value is None or expected_value is None:
        return 0.0

    # Return 1.0 if they match, 0.0 if they don't
    return 1.0 if output_value == expected_value else 0.0
'''

metric_id = client.create_metric(
    project_id=project_id,
    code=metric_code,
    metric_name="Perpetrator Classification Accuracy",
    metric_description="Measures whether the model correctly identifies perpetrators based on article evidence"
)

print(f"✓ Created metric: {metric_id}")

## Step 7: Run Tuning

Start the prompt tuning session. The model will receive prompts with template variables substituted.

In [ ]:
tuning_result = client.run_tuning(
    project_id=project_id,
    dataset_id=dataset_id,
    prompt_id=prompt_id,
    metric_id=metric_id,
    num_iterations=3,
    model_name="gpt-4o",
    provider="openai"
)

session_id = tuning_result["tuning_session_id"]
task_id = tuning_result["task_id"]

print(f"✓ Tuning started")
print(f"  Session ID: {session_id}")
print(f"  Task ID: {task_id}")

## Step 8: Wait for Completion

Poll the tuning session until it completes. This can take 10-30 minutes.

In [ ]:
# Optional: Enable DEBUG logging to see polling progress
# logging.getLogger().setLevel(logging.DEBUG)

final_result = client.wait_for_tuning_completion(
    session_id=session_id,
    poll_interval=15,
    timeout=3600
)

print("\n=== Final Results ===")
print(f"Session ID: {final_result['session_id']}")
print(f"Status: {final_result['status']}")
print(f"Best Metric Value: {final_result.get('best_metric_value', 'N/A')}")
print(f"Iterations: {final_result.get('max_iterations', 'N/A')}")

if final_result.get("metrics"):
    print("\nMetrics:")
    print(json.dumps(final_result["metrics"], indent=2))

## Step 9: Fetch Best Prompt

In [ ]:
best_prompt = client.get_best_prompt(session_id)

print("\n=== Best Prompt ===")
print(f"Prompt ID: {best_prompt['prompt_id']}")
print(f"Prompt Name: {best_prompt.get('name', 'N/A')}")
print(f"\nMessages:")
for i, msg in enumerate(best_prompt.get('messages', []), 1):
    print(f"\n--- Message {i} ({msg.get('prompt_message_type', 'unknown')}) ---")
    print(msg.get('prompt_text', ''))

if best_prompt.get('output_schema'):
    print(f"\nOutput Schema:")
    print(json.dumps(best_prompt['output_schema'], indent=2))

## Summary

You've successfully completed template-based prompt tuning with the DimRed API:

- ✓ Loaded template dataset with template_var payloads
- ✓ Created project and dataset
- ✓ Added datapoints with template variable payloads
- ✓ Created prompt with `{{customer_name}}` placeholder
- ✓ Created classification metric
- ✓ Ran tuning with template variable substitution
- ✓ Retrieved optimized prompt

## Key Points for Template Variable Payloads

1. **Template Payload Structure**: Use `payload_type: "template_var"` with `{key, value}` format
2. **Placeholder Syntax**: Use double curly braces in prompts: `{{variable_name}}`
3. **Substitution**: The system automatically replaces placeholders with payload values
4. **Model Selection**: Any text model works (no special capabilities required)
5. **Multiple Variables**: You can have multiple template_var payloads per datapoint
6. **Use Case**: Ideal for testing the same prompt template with different variable values